# Análisis de Word2Vec en reseñas de películas

BoW, TF-IDF, CBOW y Skip-Gram usando las 12.368 reseñas originales.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
from sklearn.decomposition import PCA
from gensim.models import Word2Vec
RAIZ = Path.cwd()
if not (RAIZ / "resultados").exists():
    raise FileNotFoundError("Abra Jupyter desde la carpeta del proyecto.")
df = pd.read_csv(RAIZ / "data" / "resenas_mongodb.csv")
ids = pd.read_csv(RAIZ / "resultados" / "ids_resenas.csv")
bow = load_npz(RAIZ / "resultados" / "bow.npz")
tfidf = load_npz(RAIZ / "resultados" / "tfidf.npz")
cbow = pd.read_csv(RAIZ / "resultados" / "vectores_cbow.csv.gz")
sg = pd.read_csv(RAIZ / "resultados" / "vectores_skip_gram.csv.gz")
assert len(df)==len(ids)==len(cbow)==len(sg)
assert ids.id_resena.equals(cbow.id_resena) and ids.id_resena.equals(sg.id_resena)
print("Reseñas:",len(df),"BoW:",bow.shape,"TF-IDF:",tfidf.shape)

In [ ]:
print(df[["genero_principal","polaridad","calificacion"]].describe(include="all"))
fig,axes=plt.subplots(1,2,figsize=(13,4))
df["polaridad"].fillna("Sin dato").value_counts().plot.bar(ax=axes[0],title="Polaridad")
df["genero_principal"].fillna("Sin dato").value_counts().head(10).plot.bar(ax=axes[1],title="Géneros más frecuentes")
plt.tight_layout(); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].hist(df["metricas.num_palabras"].dropna(),bins=40); axes[0].set_title("Longitud de las reseñas")
axes[1].hist(df["calificacion"].dropna(),bins=10); axes[1].set_title("Calificaciones")
plt.tight_layout();plt.show()

In [ ]:
modelo_cbow=Word2Vec.load(str(RAIZ/"resultados"/"modelo_cbow.model"))
modelo_sg=Word2Vec.load(str(RAIZ/"resultados"/"modelo_skip_gram.model"))
for nombre,modelo in [("CBOW",modelo_cbow),("Skip-Gram",modelo_sg)]:
    print("\n",nombre,"| vocabulario:",len(modelo.wv))
    for palabra in ["película","historia","actor"]:
        if palabra in modelo.wv:
            print(palabra,modelo.wv.most_similar(palabra,topn=5))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,(nombre,tabla) in zip(axes,[("CBOW",cbow),("Skip-Gram",sg)]):
    coords=PCA(n_components=2,random_state=42).fit_transform(tabla.iloc[:1000,1:].to_numpy())
    ax.scatter(coords[:,0],coords[:,1],s=6,alpha=.5)
    ax.set_title(f"PCA de vectores de reseñas: {nombre} (muestra de 1000)")
plt.tight_layout();plt.show()

## Interpretación

- **BoW** registra frecuencias de términos, sin considerar contexto.
- **TF-IDF** pondera la importancia relativa de términos.
- **CBOW** predice palabras a partir de su contexto.
- **Skip-Gram** predice palabras del contexto a partir de una palabra central.
- Los vectores de reseña se calculan promediando los vectores de sus palabras; son una representación simplificada del documento.
- PCA permite visualizar relaciones aproximadas en dos dimensiones; no demuestra por sí solo calidad predictiva.